# AMD GPU + Build Spec Starter

Run this notebook inside AMD Developer Cloud Jupyter. It verifies ROCm/PyTorch GPU access, optionally runs a small Qwen model, and generates the first business build specs for the hackathon project.

In [ ]:
!rocm-smi

In [ ]:
import torch

print('Torch:', torch.__version__)
print('CUDA API available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device count:', torch.cuda.device_count())
    print('Device 0:', torch.cuda.get_device_name(0))
else:
    print('No GPU visible to PyTorch. On ROCm builds, torch.cuda should still be True.')

## Optional: Small Qwen Inference Test

This downloads a small model. If the download is slow or blocked, skip this cell and continue to the deterministic planner.

In [ ]:
# Optional install if your environment does not already have these packages.
# !pip install -q transformers accelerate sentencepiece

from transformers import pipeline

if torch.cuda.is_available():
    generator = pipeline(
        'text-generation',
        model='Qwen/Qwen2.5-0.5B-Instruct',
        device=0,
    )
    prompt = 'Generate a concise homepage headline for a dental clinic called BrightCare Dental.'
    output = generator(prompt, max_new_tokens=60, do_sample=False)
    print(output[0]['generated_text'])
else:
    print('Skipping model test because no GPU is visible.')

## Build Spec Generation

This is the backbone of the product: raw business input becomes a structured plan that decides pages, backend modules, trust/compliance checks, QA tests, and readiness scores.

In [ ]:
import json
from buildspec_planner import generate_build_spec

restaurant_profile = {
    'name': 'Bella Napoli',
    'location': 'San Francisco',
    'goal': 'increase online orders and table reservations',
    'contact_email': 'hello@bellanapoli.example',
}

restaurant_input = '''
Bella Napoli is a family Italian restaurant in San Francisco.
It serves pizza, pasta, desserts, and has a menu for pickup orders.
Business hours are 11am to 10pm daily.
The owner wants more online orders and table reservations.
'''

restaurant_spec = generate_build_spec(restaurant_profile, restaurant_input)
print(json.dumps(restaurant_spec, indent=2))

In [ ]:
clinic_profile = {
    'name': 'BrightCare Dental',
    'location': 'Austin',
    'goal': 'increase appointment bookings',
    'contact_email': 'appointments@brightcare.example',
}

clinic_input = '''
BrightCare Dental is a dental clinic in Austin.
It offers cleaning, whitening, emergency dental care, implants, and family dentistry.
Business hours are Monday to Friday 9am to 6pm.
Patients should be able to book appointments online and submit intake information.
'''

clinic_spec = generate_build_spec(clinic_profile, clinic_input)
print(json.dumps(clinic_spec, indent=2))

## Export For Local App

Run this cell, then open `restaurant-buildspec.json` or `clinic-buildspec.json` and paste the JSON into the AMD Proof tab of the local browser app.

In [ ]:
from pathlib import Path

Path('restaurant-buildspec.json').write_text(json.dumps(restaurant_spec, indent=2), encoding='utf-8')
Path('clinic-buildspec.json').write_text(json.dumps(clinic_spec, indent=2), encoding='utf-8')

print('Wrote restaurant-buildspec.json and clinic-buildspec.json')

## AMD Vision / Multimodal Asset Extraction

Upload menu, flyer, brochure, storefront, or service-list images into this notebook folder. Then install the vision dependencies and run the extraction cells below. Qwen2.5-VL reads the images and produces structured business signals for the planner.

In [ ]:
# Run once if the vision dependencies are not installed.
# This can take a few minutes because it installs a recent Transformers build.
# !pip install -r requirements-amd-vision.txt

In [ ]:
from pathlib import Path

image_paths = sorted(
    [*Path('.').glob('*.jpg'), *Path('.').glob('*.jpeg'), *Path('.').glob('*.png'), *Path('.').glob('*.webp')]
)
print('Images found:')
for path in image_paths:
    print('-', path)

if not image_paths:
    print('Upload image files into this folder, then rerun this cell.')

In [ ]:
# Load Qwen2.5-VL on AMD GPU.
# On ROCm, PyTorch still exposes the GPU through the torch.cuda API.
from amd_vision_extract import extract_asset_info, load_model, summarize_for_buildspec

vision_model, vision_processor = load_model()
print('Loaded vision model on AMD GPU')

In [ ]:
asset_extractions = []
for path in image_paths:
    print(f'Extracting: {path}')
    asset_extractions.append(extract_asset_info(path, vision_model, vision_processor))

asset_signals_text = summarize_for_buildspec(asset_extractions)
print(asset_signals_text)

Path('asset-extractions.json').write_text(json.dumps(asset_extractions, indent=2), encoding='utf-8')
Path('asset-signals.txt').write_text(asset_signals_text, encoding='utf-8')
print('Wrote asset-extractions.json and asset-signals.txt')

### Use Vision Output In BuildSpec

Append `asset_signals_text` to the business input before calling `generate_build_spec(...)`. This makes the same planner choose features based on the uploaded image evidence.

In [ ]:
# Example: enrich restaurant input with extracted image signals.
if 'asset_signals_text' in globals() and asset_signals_text.strip():
    enriched_restaurant_spec = generate_build_spec(
        restaurant_profile,
        restaurant_input + '\n\n' + asset_signals_text,
    )
    Path('restaurant-buildspec-from-vision.json').write_text(
        json.dumps(enriched_restaurant_spec, indent=2),
        encoding='utf-8',
    )
    print(json.dumps(enriched_restaurant_spec, indent=2))
    print('Wrote restaurant-buildspec-from-vision.json')

## Next Integration Target

Once the notebook works, move `generate_build_spec(...)` behind an app endpoint:

`Frontend intake form -> /api/generate-spec -> BuildSpec JSON -> website/backend generator -> QA report`